# Projekt 8

## Filip Nocoń
---

### Inicjalizacja potrzebnych bibliotek

In [1]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
%matplotlib inline

----
### 1. Wczytanie plików z danymi. Inspekcja zawartości plików 

In [2]:
df = pd.read_csv("dataset_13.csv")
print(f"Rozmiar ramki: {df.shape}")
print(f"Kolumny: {df.columns.tolist()}")
print("************************************************************************************")
print(df.describe())
print("************************************************************************************")

df2 = pd.read_csv("dataset_33.csv")
print(f"Rozmiar ramki: {df2.shape}")
print(f"Kolumny: {df2.columns.tolist()}")
print("************************************************************************************")
print(df2.describe())
print("************************************************************************************")

Rozmiar ramki: (1100, 11)
Kolumny: ['x_0', 'x_1', 'x_2', 'x_3', 'x_4', 'x_5', 'x_6', 'x_7', 'x_8', 'x_9', 'y']
************************************************************************************
               x_0          x_1          x_2          x_3          x_4  \
count  1100.000000  1100.000000  1100.000000  1100.000000  1100.000000   
mean      1.465274    -0.889791     2.642457    -0.010650     1.009388   
std       1.002913     1.745907     1.191256     1.053991     1.782497   
min      -1.749317    -5.461661    -1.191391    -3.488652    -3.931896   
25%       0.786296    -2.129460     1.817101    -0.712252    -0.283353   
50%       1.454152    -1.183529     2.670914    -0.004741     1.113231   
75%       2.151374     0.023618     3.443645     0.710136     2.306624   
max       4.965341     5.470456     6.035149     4.044528     6.032771   

               x_5          x_6          x_7          x_8          x_9  \
count  1100.000000  1100.000000  1100.000000  1100.000000  1100

---
### 2. Określenie liczebności poszczególnych klas; badanie ich zrównowoważenia55

In [3]:
print("Liczebność klas binarne:", Counter(df['y']))
print("Udziały klas:\n", df['y'].value_counts(normalize=True))
print("************************************************************************************")
print("Liczebność klas multi:", Counter(df2['y']))
print("Udziały klas:\n", df2['y'].value_counts(normalize=True))

Liczebność klas binarne: Counter({0: 905, 1: 195})
Udziały klas:
 y
0    0.822727
1    0.177273
Name: proportion, dtype: float64
************************************************************************************
Liczebność klas multi: Counter({0: 409, 1: 246, 3: 182, 2: 163})
Udziały klas:
 y
0    0.409
1    0.246
3    0.182
2    0.163
Name: proportion, dtype: float64


---
### 3. Przygotowanie macierzy cech oraz wektorów etykiet

In [4]:
target_column = 'y' 

X = df.drop(columns=[target_column])
y = df[target_column]

print("Typy danych w macierzy cech X:")
print(X.dtypes)
print("\nTyp danych wektora etykiet y:")
print(y.dtype)
print("************************************************************************************")
X2 = df2.drop(columns=[target_column])
y2 = df2[target_column]

print("Typy danych w macierzy cech X:")
print(X2.dtypes)
print("\nTyp danych wektora etykiet y:")
print(y2.dtype)

Typy danych w macierzy cech X:
x_0    float64
x_1    float64
x_2    float64
x_3    float64
x_4    float64
x_5    float64
x_6    float64
x_7    float64
x_8    float64
x_9    float64
dtype: object

Typ danych wektora etykiet y:
int64
************************************************************************************
Typy danych w macierzy cech X:
x_0    float64
x_1    float64
x_2    float64
x_3    float64
x_4    float64
x_5    float64
dtype: object

Typ danych wektora etykiet y:
int64


### 4. Podział zbioru danych na zbór treningowy i testowy: prosty i ze stratyfikacją

In [5]:
X = df.drop(columns=['y'])
y = df['y']

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X, y, test_size=0.3, random_state=42)

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

print("Rozkład w y_test (prosty):", Counter(y_test_p))
print("Rozkład w y_test (stratyfikowany):", Counter(y_test_s))
print("************************************************************************************")
X2 = df2.drop(columns=['y'])
y2 = df2['y']

X2_train_p, X2_test_p, y2_train_p, y2_test_p = train_test_split(X2, y2, test_size=0.3, random_state=42)

X2_train_s, X2_test_s, y2_train_s, y2_test_s = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=42)
print("Rozkład w y_test (prosty):", Counter(y2_test_p))
print("Rozkład w y_test (stratyfikowany):", Counter(y2_test_s))


Rozkład w y_test (prosty): Counter({0: 274, 1: 56})
Rozkład w y_test (stratyfikowany): Counter({0: 272, 1: 58})
************************************************************************************
Rozkład w y_test (prosty): Counter({0: 112, 1: 69, 2: 60, 3: 59})
Rozkład w y_test (stratyfikowany): Counter({0: 123, 1: 74, 3: 54, 2: 49})


---
### 5a. Badanie klasyfikatorów binarnych: LDA, SVM, regresja logistyczna - prosty podział na zbiór treningowy i testowy

In [6]:
modele_binarne = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVM_linear": SVC(kernel='linear'),
    "Regresja_Logistyczna": LogisticRegression()
}

for nazwa, model in modele_binarne.items():
    model.fit(X_train_p, y_train_p)
    y_pred = model.predict(X_test_p)
    print(classification_report(y_test_p, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00       274
           1       0.98      0.98      0.98        56

    accuracy                           0.99       330
   macro avg       0.99      0.99      0.99       330
weighted avg       0.99      0.99      0.99       330

              precision    recall  f1-score   support

           0       1.00      0.99      0.99       274
           1       0.96      0.98      0.97        56

    accuracy                           0.99       330
   macro avg       0.98      0.99      0.98       330
weighted avg       0.99      0.99      0.99       330

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       274
           1       0.97      1.00      0.98        56

    accuracy                           0.99       330
   macro avg       0.98      1.00      0.99       330
weighted avg       0.99      0.99      0.99       330



---
### 5b. Badanie klasyfikatorów binarnych: LDA, SVM, regresja logistyczna - podział stratyfikowany na zbiór treningowy i testowy

In [7]:
for nazwa, model in modele_binarne.items():
    model.fit(X_train_s, y_train_s)
    y_pred = model.predict(X_test_s)
    print(classification_report(y_test_s, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       272
           1       0.94      0.84      0.89        58

    accuracy                           0.96       330
   macro avg       0.95      0.92      0.93       330
weighted avg       0.96      0.96      0.96       330

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       272
           1       0.98      0.97      0.97        58

    accuracy                           0.99       330
   macro avg       0.99      0.98      0.98       330
weighted avg       0.99      0.99      0.99       330

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       272
           1       0.98      0.93      0.96        58

    accuracy                           0.98       330
   macro avg       0.98      0.96      0.97       330
weighted avg       0.98      0.98      0.98       330



---
### 6a. Badanie klasyfikatorów wieloklasowych: LDA, SVM, las losowy - prosty podział na zbiór treningowy i testowy

In [8]:
models_multi = {
    "LDA": LinearDiscriminantAnalysis(),
    "SVM_rbf": SVC(kernel='rbf'),
    "RandomForest": RandomForestClassifier()
}

for name, model in models_multi.items():
    model.fit(X2_train_p, y2_train_p)
    preds = model.predict(X2_test_p)
    print(f"Wyniki wieloklasowe dla {name}:\n", classification_report(y2_test_p, preds))

Wyniki wieloklasowe dla LDA:
               precision    recall  f1-score   support

           0       0.96      0.96      0.96       112
           1       0.92      0.96      0.94        69
           2       0.95      0.93      0.94        60
           3       0.98      0.95      0.97        59

    accuracy                           0.95       300
   macro avg       0.95      0.95      0.95       300
weighted avg       0.95      0.95      0.95       300

Wyniki wieloklasowe dla SVM_rbf:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98       112
           1       0.97      0.99      0.98        69
           2       0.98      0.95      0.97        60
           3       0.95      0.97      0.96        59

    accuracy                           0.97       300
   macro avg       0.97      0.97      0.97       300
weighted avg       0.97      0.97      0.97       300

Wyniki wieloklasowe dla RandomForest:
               precision    

---
### 6b. Badanie klasyfikatorów wieloklasowych: LDA, SVM, las losowy - podział stratyfikowany na zbiór treningowy i testowy

In [9]:
for name, model in models_multi.items():
    model.fit(X2_train_s, y2_train_s)
    preds = model.predict(X2_test_s)
    print(f"Wyniki wieloklasowe dla {name}:\n", classification_report(y2_test_s, preds))

Wyniki wieloklasowe dla LDA:
               precision    recall  f1-score   support

           0       0.94      0.98      0.96       123
           1       0.91      0.95      0.93        74
           2       0.98      0.90      0.94        49
           3       0.98      0.93      0.95        54

    accuracy                           0.95       300
   macro avg       0.95      0.94      0.94       300
weighted avg       0.95      0.95      0.95       300

Wyniki wieloklasowe dla SVM_rbf:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       123
           1       0.99      0.97      0.98        74
           2       0.94      0.98      0.96        49
           3       0.98      0.94      0.96        54

    accuracy                           0.98       300
   macro avg       0.97      0.97      0.97       300
weighted avg       0.98      0.98      0.98       300

Wyniki wieloklasowe dla RandomForest:
               precision    

#### Który model należy wybrać dla klasyfikacji binarnej i wieloklasowej i z jakich powodów? Skomentować wpływ  niezrównoważenia klas oraz sposobu podziału danych na metryki klasyfikacji.

###
Binarny:
- przy prostej wszystkie radzą sobie podobnie.
- przy stratyfikowanej SVM radzi sobie najlepiej
  
Wieloklasowy:
- proste i strattfikowane najelpiej radzi sobie randomforest

Z obserwacji wynika że wyniki na zbiorze stratyfikowanym są bardziej powtarzalne, podczas dzielenia przydzielania danych treningowych, dawały podobne wyniki. A podczas prostego raz mi się zdarzyło że LDA było bardzo niskie, około 90


In [10]:
# Eksportowanie bieżacego notatnika do pdf.

%run ../data/notebook_export
export_notebook_to_pdf("Projekt8_Nocoń_Filip.ipynb")

Export 'Projekt8_Nocoń_Filip.ipynb': OK.


**PDF wygenerowano:** Wednesday, 14 January 2026 13:47:28